# Kwartalny wstępny rachunek zysków i strat z użyciem PROC COMPUTAB

## Streszczenie kierownicze

Ten notatnik tworzy kwartalny wstępny (pro forma) rachunek zysków i strat regionalnego banku bezpośrednio z miesięcznych danych księgi głównej za pomocą PROC COMPUTAB — procedury tabelarycznego raportowania SAS/ETS. Kieruje przychody odsetkowe, koszty odsetkowe, przychody z opłat oraz koszty operacyjne każdego miesiąca do właściwej kolumny kwartału kalendarzowego, a następnie za pomocą bloków programowania wierszy oblicza wynik z tytułu odsetek netto, dochód przed opodatkowaniem i dochód netto, natomiast blok kolumn sumuje cztery kwartały w łączną wartość roczną.

## Źródła danych

Jeden syntetyczny zbiór danych `bank_ledger` symuluje jeden rok obrotowy miesięcznych pozycji sprawozdania finansowego dla średniej wielkości banku spółdzielczego. Dwanaście miesięcznych obserwacji jest generowanych w miejscu za pomocą `call streaminit`/`rand`, dzięki czemu notatnik jest w pełni samowystarczalny.

| Zmienna | Typ | Opis |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Data sprawozdania na koniec miesiąca (sty–gru) |
| `loanint`  | Num | Odsetki i opłaty uzyskane z portfela kredytowego (tys. USD) |
| `secint`   | Num | Odsetki uzyskane z portfela papierów wartościowych (tys. USD) |
| `depint`   | Num | Odsetki zapłacone od depozytów (tys. USD) |
| `borrint`  | Num | Odsetki zapłacone od zobowiązań / zaliczek FHLB (tys. USD) |
| `feeinc`   | Num | Przychody pozaodsetkowe (opłaty i prowizje za usługi) (tys. USD) |
| `salaries` | Num | Koszty wynagrodzeń i świadczeń pracowniczych (tys. USD) |
| `occupancy`| Num | Koszty utrzymania nieruchomości i wyposażenia (tys. USD) |
| `othropex` | Num | Pozostałe pozaodsetkowe koszty operacyjne (tys. USD) |
| `provision`| Num | Rezerwa na straty kredytowe (tys. USD) |
| `taxrate`  | Num | Efektywna stawka podatkowa stosowana do dochodu przed opodatkowaniem |

# Kwartalny wstępny rachunek zysków i strat z użyciem PROC COMPUTAB

Zespoły finansowe banków rutynowo przekształcają miesięczną księgę główną w **kwartalny rachunek zysków i strat**, który pokazuje wynik z tytułu odsetek netto oraz wynik netto na dole zestawienia. `PROC COMPUTAB` (SAS/ETS) jest do tego stworzony: układa programowalną tabelę, której **kolumny** są okresami sprawozdawczymi, a **wiersze** pozycjami zestawienia, i pozwala obliczać podsumowania za pomocą zwykłej logiki kroku DATA w blokach wierszy i kolumn.

W tym notatniku:

1. Generujemy jeden rok obrotowy syntetycznych miesięcznych danych księgi głównej banku spółdzielczego.
2. Kierujemy każdy miesiąc do jego kwartału kalendarzowego i budujemy kwartalny rachunek zysków i strat.
3. Obliczamy wynik z tytułu odsetek netto, dochód przed opodatkowaniem i dochód netto w **bloku wierszy**, a następnie sumujemy kwartały w łączną wartość roczną w **bloku kolumn**.
4. Ponownie wykorzystujemy tabelę `out=`, aby obliczone zestawienie mogło zasilać dalsze raportowanie.

## Krok 1 — Wygenerowanie syntetycznych miesięcznych danych księgi głównej

Symulujemy dwanaście obserwacji na koniec miesiąca. Przychody odsetkowe z kredytów i papierów wartościowych łagodnie rosną w ciągu roku, koszty depozytów i zobowiązań skalują się wraz z otoczeniem stóp procentowych, a przychody z opłat, koszty operacyjne i rezerwa na straty kredytowe niosą realistyczny szum z miesiąca na miesiąc. `call streaminit` ustala ziarno, dzięki czemu dane są odtwarzalne.

In [1]:
DANE bank_ledger;
   CALL streaminit(20240115);
   format stmtdate date9.;
   POWTÓRZ month = 1 TO 12;
      /* Data sprawozdania na koniec miesiąca dla roku obrotowego 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Łagodny wzrostowy trend w ciągu roku + miesięczny szum */
      drift = 1 + 0.012 * (month - 1);

      /* Przychody odsetkowe (tys. USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Koszty odsetkowe (tys. USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Przychody i koszty pozaodsetkowe (tys. USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Rezerwa na straty kredytowe, okazjonalnie podwyższona */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Efektywna stawka podatkowa */
      taxrate = 0.21;

      WYJŚCIE;
   KONIEC;
   ZACHOWAJ stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
   ETYKIETA stmtdate="Data sprawozdania"
            loanint="Odsetki i opłaty od kredytów"
            secint="Odsetki od papierów wartościowych"
            depint="Odsetki od depozytów"
            borrint="Odsetki od zobowiązań"
            feeinc="Przychody z opłat"
            salaries="Wynagrodzenia i świadczenia"
            occupancy="Nieruchomości i wyposażenie"
            othropex="Pozostałe koszty operacyjne"
            provision="Rezerwa na straty kredytowe"
            taxrate="Stawka podatkowa";
WYKONAJ;

PROCEDURA DRUKUJ DANE=bank_ledger noobs ETYKIETA;
   TYTUŁ "Syntetyczna miesięczna księga główna — bank spółdzielczy, rok obr. 2024";
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
WYKONAJ;

                        Syntetyczna miesięczna księga główna — bank spółdzielczy, rok obr. 2024                         

Data sprawozdania    Odsetki i opłaty od kredytów    Odsetki od papierów wartościowych   Odsetki od depozytów    Odsetki od zobowiązań   Przychody z opłat   Wynagrodzenia i świadczenia    Nieruchomości i wyposażenie   Pozostałe koszty operacyjne  Rezerwa na straty kredytowe  Stawka podatkowa
        28JAN2024                        1,772.95                               423.79                 531.47                   128.99              339.33                        699.38                         171.95                        202.43                       130.93              0.21
        28FEB2024                        1,824.38                               421.13                 564.85                   125.53              294.09                        767.29                         162.79                        307.61                       123.25              0.21


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Krok 2 — Zbudowanie kwartalnego rachunku zysków i strat

Sercem raportu jest poniższy krok `PROC COMPUTAB`.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definiuje cztery kolumny kwartałów oraz kolumnę pełnego roku.
- Nieoznaczony **blok wejściowy** ustawia zmienną automatyczną **`_col_`** za pomocą `qtr(stmtdate)`, co kieruje każdą miesięczną obserwację do właściwej kolumny kwartału. Ponieważ dane wejściowe są domyślnie transponowane, każda zmienna księgi trafia do wiersza o tej samej nazwie.
- **Blok wierszy** `inc_stmt:` uruchamia się raz na kolumnę i oblicza pozycje pochodne — wynik z tytułu odsetek netto, łączne koszty pozaodsetkowe, dochód przed opodatkowaniem i dochód netto — dokładnie tak, jak zrobiłby to księgowy.
- **Blok kolumn** `total:` uruchamia się raz na wiersz i sumuje cztery kwartały do kolumny `fy` (pełny rok).

Instrukcje `rows ... / ...` dodają tytuły, wcięcia i linie reguł (`ol` nadkreślenie, `ul` podkreślenie, `dul` podwójne podkreślenie), dzięki czemu wynik czyta się jak prawdziwe sprawozdanie finansowe.

In [2]:
TYTUŁ "Wstępny rachunek zysków i strat — Community Bancorp, Inc.";
title2 "Rok obrotowy 2024  (kwoty w tys. USD)";

PROCEDURA computab DANE=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Cztery kwartały plus zsumowana kolumna pełnego roku */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Kw1';
   columns qtr2 / 'Kw2';
   columns qtr3 / 'Kw3';
   columns qtr4 / 'Kw4';
   columns fy   / 'Pełny' 'Rok' +3;

   /* Wiersze rachunku zysków i strat, od góry do dołu */
   rows loanint  / 'Odsetki i opłaty od kredytów';
   rows secint   / 'Odsetki od papierów wartościowych'   ul;
   rows intinc   / 'Łączne przychody odsetkowe';
   rows depint   / 'Odsetki od depozytów';
   rows borrint  / 'Odsetki od zobowiązań'               ul;
   rows intexp   / 'Łączne koszty odsetkowe';
   rows nii      / 'Wynik z tytułu odsetek netto'        ol skip;
   rows provision/ 'Rezerwa na straty kredytowe'         ul;
   rows niiap    / 'Wynik odsetkowy netto po rezerwie'   skip;
   rows feeinc   / 'Przychody pozaodsetkowe'             ul;
   rows salaries / 'Wynagrodzenia i świadczenia';
   rows occupancy/ 'Nieruchomości i wyposażenie';
   rows othropex / 'Pozostałe koszty operacyjne'         ul;
   rows nonintexp/ 'Łączne koszty pozaodsetkowe'         skip;
   rows pretax   / 'Dochód przed opodatkowaniem'         ol;
   rows taxexp   / 'Podatek dochodowy'                   ul;
   rows netinc   / 'Dochód netto'                        dul;

   /* Blok wejściowy: kieruj każdy miesiąc do jego kwartału kalendarzowego */
   _col_ = qtr(stmtdate);

   /* Blok wierszy: oblicz podsumowania zestawienia we wszystkich kolumnach */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Blok kolumn: zsumuj cztery kwartały do kolumny pełnego roku */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
WYKONAJ;

                               Wstępny rachunek zysków i strat — Community Bancorp, Inc.                                
                                         Rok obrotowy 2024  (kwoty w tys. USD)                                          


                             The COMPUTAB Procedure                             

                            Kw1          Kw2          Kw3          Kw4        Pełny  
                                                                                Rok  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Odsetki i opłaty od kredytów
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Odsetki od papierów wartościowych
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Łąc


NOTE: Option TITLE changed to Wstępny rachunek zysków i strat — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Rok obrotowy 2024  (kwoty w tys. USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Krok 3 — Ponowne wykorzystanie zbioru wynikowego COMPUTAB

Powyższy krok `PROC COMPUTAB` zapisał obliczoną tabelę do `qtr_income` za pomocą `out=`. Każdy wiersz tego zbioru danych jest pozycją zestawienia, a każda zmienna kolumnowa (`qtr1`–`qtr4`, `fy`) przechowuje obliczoną wartość, więc może zasilać dalsze raportowanie. Poniżej drukujemy zsumowaną kolumnę pełnego roku, aby potwierdzić, że liczby się zgadzają.

In [3]:
TYTUŁ "Zbiór wynikowy COMPUTAB — kolumna pełnego roku";

PROCEDURA DRUKUJ DANE=qtr_income ETYKIETA noobs;
   ZMIENNA _row_ fy;
   format fy comma12.1;
   ETYKIETA _row_ = "Pozycja zestawienia" fy = "Pełny rok";
WYKONAJ;

TYTUŁ;

                                     Zbiór wynikowy COMPUTAB — kolumna pełnego roku                                     
                                         Rok obrotowy 2024  (kwoty w tys. USD)                                          


Pozycja zestawienia   Pełny rok
-------------------  ----------
loanint                23,588.4
secint                  5,416.8
intinc                 29,005.2
depint                  6,831.2
borrint                 1,650.7
intexp                  8,482.0
nii                    20,523.2
provision               1,568.9
niiap                  18,954.3
feeinc                  3,703.2
salaries                8,763.1
occupancy               1,985.6
othropex                2,944.8
nonintexp              13,693.5
pretax                  8,964.1
taxexp                  1,882.5
netinc                  7,081.6




NOTE: Option TITLE changed to Zbiór wynikowy COMPUTAB — kolumna pełnego roku.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretacja wyników

Wstępny rachunek zysków i strat czyta się od góry do dołu jak regulacyjny raport sprawozdawczy (call report) banku: łączne przychody odsetkowe pomniejszone o łączne koszty odsetkowe dają **wynik z tytułu odsetek netto** — tutaj około 20,5 mln USD za rok — główny czynnik zysków banku. Odjęcie **rezerwy na straty kredytowe**, dodanie **przychodów z opłat** i odjęcie **kosztów pozaodsetkowych** daje dochód przed opodatkowaniem w wysokości około 9,0 mln USD, a zastosowanie efektywnej stawki podatkowej 21% daje **dochód netto** bliski 7,1 mln USD. Blok kolumn `total:` sumuje cztery kwartały do kolumny pełnego roku, więc łączne wartości roczne z założenia uzgadniają się z sumą kwartałów — co potwierdza zbiór `out=`, w którym roczny `netinc` wynoszący 7 081,6 równa się sumie czterech wartości kwartalnych.

Ponieważ wszystko jest obliczane wewnątrz programowalnych bloków wierszy i kolumn `PROC COMPUTAB`, podstawienie prawdziwej miesięcznej księgi głównej nie wymaga żadnej zmiany logiki raportu — zmienia się jedynie zbiór wejściowy. Zbiór `out=` udostępnia następnie obliczone zestawienie do pulpitów menedżerskich, analizy trendów lub automatyzacji pakietów dla zarządu.